# Velocity fit of fσ₈ using the simulation forward model

This notebook demonstrates how to use the `flip.simulation` forward-model pipeline to
fit the growth parameter **fσ₈** from a peculiar velocity catalogue.

The forward model uses a differentiable **particle-mesh simulation** (LPT or full N-body
via `JaxPM` + `diffrax`) and a gradient-based optimizer (`jaxopt`) to maximise the
likelihood of the observed velocities with respect to the cosmological parameters.

**Outline:**
1. Install dependencies
2. Set up the simulation box and generate a synthetic mock catalogue
3. Compute the true fσ₈ from the input cosmology
4. Build the `VelocityFieldLikelihood` and scan the likelihood over σ8
5. Fit σ8 (and fσ₈) with `SimulationFitter` using gradient descent
6. Repeat with the full N-body ODE solver

> **Note:** This notebook uses a small mesh (16³) to run in minutes on CPU.
> For a science-quality analysis use at least a 128³ mesh.

In [ ]:
%%capture
!pip install git+https://github.com/corentinravoux/flip "jaxpm>=0.1" "diffrax>=0.5" jax_cosmo jaxopt

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from flip import data_vector, __flip_dir_path__
from flip.simulation import generate, likelihood
from flip.simulation.fitter import SimulationFitter

# Enable 64-bit precision for accurate simulation
jax.config.update("jax_enable_x64", True)

flip_base = Path(__flip_dir_path__)
data_path = flip_base / "data"
plt.style.use(data_path / "style.mplstyle")

print("JAX devices:", jax.devices())

## 1. Simulation box setup

Choose a box large enough to contain the galaxy sample.
We use a 16³ mesh for speed; a production analysis would use 128³ or larger.

In [ ]:
# -----------------------------------------------------------------------
# Simulation parameters
# -----------------------------------------------------------------------
MESH_SHAPE = (16, 16, 16)       # grid cells per axis (increase for science)
BOX_SIZE   = [128., 128., 128.]  # Mpc/h per axis
SEED       = jax.random.PRNGKey(0)

# -----------------------------------------------------------------------
# True (fiducial) cosmology used to create the mock catalogue
# -----------------------------------------------------------------------
TRUE_OMEGA_M = 0.3
TRUE_SIGMA8  = 0.8

cosmo_true = generate.get_cosmology(omega_m=TRUE_OMEGA_M, sigma8=TRUE_SIGMA8)
print("True cosmology:")
print(f"  Omega_m = {TRUE_OMEGA_M}")
print(f"  sigma_8 = {float(cosmo_true.sigma8):.4f}")

## 2. Generate a synthetic mock velocity catalogue

We simulate the velocity field with LPT and paint mock galaxies at random positions.
Each galaxy receives the true simulated velocity plus Gaussian measurement noise.

In [ ]:
# -----------------------------------------------------------------------
# True velocity field (LPT for mock generation)
# -----------------------------------------------------------------------
_, vel_field_true = generate.generate_density_and_velocity_lpt(
    cosmo_true, MESH_SHAPE, BOX_SIZE, SEED
)

# -----------------------------------------------------------------------
# Random galaxy positions inside the box [Mpc/h]
# -----------------------------------------------------------------------
N_GALAXIES     = 150
VELOCITY_ERROR = 200.0  # km/s per galaxy

rng = np.random.RandomState(42)
positions = rng.uniform(5.0, float(BOX_SIZE[0]) - 5.0, (N_GALAXIES, 3))

# Interpolate true velocity at galaxy positions and project onto LOS
vel_3d_true  = generate.interpolate_velocity_to_positions(
    vel_field_true, jnp.array(positions), jnp.array(BOX_SIZE), MESH_SHAPE
)
los_vel_true = generate.compute_los_velocity(vel_3d_true, jnp.array(positions))

# Add Gaussian measurement noise
noise             = rng.normal(0.0, VELOCITY_ERROR, N_GALAXIES)
observed_velocities = np.array(los_vel_true) + noise

print(f"Mock catalogue: {N_GALAXIES} galaxies")
print(f"True velocity range: [{float(los_vel_true.min()):.1f}, {float(los_vel_true.max()):.1f}] km/s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = axes[0].scatter(
    positions[:, 0], positions[:, 1],
    c=np.array(los_vel_true), vmin=-400, vmax=400, cmap="RdBu_r"
)
axes[0].set_xlabel("X [Mpc/h]")
axes[0].set_ylabel("Y [Mpc/h]")
axes[0].set_title("True line-of-sight velocity")
plt.colorbar(sc, ax=axes[0], label="v_los [km/s]")

sc2 = axes[1].scatter(
    positions[:, 0], positions[:, 1],
    c=observed_velocities, vmin=-600, vmax=600, cmap="RdBu_r"
)
axes[1].set_xlabel("X [Mpc/h]")
axes[1].set_ylabel("Y [Mpc/h]")
axes[1].set_title("Observed velocity (true + noise)")
plt.colorbar(sc2, ax=axes[1], label="v_obs [km/s]")

plt.tight_layout()
plt.show()

## 3. True fσ₈

We compute the true fσ₈ from the input cosmology as a reference for the fit.

In [ ]:
fsigma8_true = float(generate.compute_fsigma8(cosmo_true, a=1.0))
print(f"True f*sigma_8 = {fsigma8_true:.4f}")
print(f"  (f = {fsigma8_true / TRUE_SIGMA8:.4f},  sigma_8 = {TRUE_SIGMA8})")

## 4. Build the DataVector and VelocityFieldLikelihood

We wrap the mock observations in a `DirectVel` data vector (following the flip naming
conventions) and pass it to `VelocityFieldLikelihood`.  We fix `omega_m` at its true
value and keep `sigma8` as the single free parameter.

In [ ]:
# Build the flip DataVector
vel_data = {
    "velocity":       observed_velocities,
    "velocity_error": np.full(N_GALAXIES, VELOCITY_ERROR),
}
DataVel = data_vector.DirectVel(vel_data)

# Build the simulation likelihood — fix omega_m, keep sigma8 free
lik_lpt = likelihood.VelocityFieldLikelihood(
    data_vector=DataVel,
    positions_cartesian=positions,
    mesh_shape=MESH_SHAPE,
    box_size=BOX_SIZE,
    seed=SEED,
    method="lpt",
    fixed_cosmo_params={"omega_m": TRUE_OMEGA_M},
)

val_true = lik_lpt({"sigma8": TRUE_SIGMA8})
print(f"Neg-log-likelihood at true sigma_8 = {TRUE_SIGMA8}: {float(val_true):.3f}")

## 5. Likelihood scan over σ8

We evaluate the likelihood at a coarse grid of σ8 values to visualise the shape of
the posterior before running the gradient-based optimizer.

In [ ]:
sigma8_grid      = np.linspace(0.4, 1.2, 10)
neg_log_lik_grid = [float(lik_lpt({"sigma8": s8})) for s8 in sigma8_grid]

# Normalise to get Delta log L
log_lik_grid  = -np.array(neg_log_lik_grid)
log_lik_grid -= log_lik_grid.max()

plt.figure(figsize=(7, 4))
plt.plot(sigma8_grid, log_lik_grid, "o-", label="log L (normalised)")
plt.axvline(TRUE_SIGMA8, color="r", ls="--", label=f"True σ₈ = {TRUE_SIGMA8}")
plt.xlabel(r"$\sigma_8$")
plt.ylabel(r"$\Delta \log \mathcal{L}$")
plt.title("Likelihood scan over σ8 (LPT forward model)")
plt.legend()
plt.tight_layout()
plt.show()

best_scan = sigma8_grid[np.argmax(log_lik_grid)]
print(f"Best σ8 from coarse scan: {best_scan:.3f}  (true: {TRUE_SIGMA8})")

## 6. Gradient-based fit with SimulationFitter (LPT)

We use `SimulationFitter` to find the maximum-likelihood σ8 via gradient descent.
The gradient flows through the entire JAX simulation graph (auto-differentiation).

In [ ]:
fitter_lpt = SimulationFitter(
    likelihood=lik_lpt,
    initial_params={"sigma8": 0.7},
    bounds=({"sigma8": 0.2}, {"sigma8": 1.6}),
    solver="LBFGSB",
    maxiter=30,
)

best_lpt    = fitter_lpt.run()
best_s8_lpt = float(best_lpt["sigma8"])

cosmo_lpt    = generate.get_cosmology(omega_m=TRUE_OMEGA_M, sigma8=best_s8_lpt)
fs8_lpt      = float(generate.compute_fsigma8(cosmo_lpt, a=1.0))

print("\n=== Best-fit results (LPT forward model) ===")
print(f"  Best-fit σ8     = {best_s8_lpt:.4f}  (true: {TRUE_SIGMA8:.4f})")
print(f"  Best-fit f*σ8   = {fs8_lpt:.4f}  (true: {fsigma8_true:.4f})")
print(f"  Relative error = {abs(best_s8_lpt - TRUE_SIGMA8) / TRUE_SIGMA8 * 100:.1f}%")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sigma8_grid, log_lik_grid, "o-", label="log L scan (LPT)")
plt.axvline(TRUE_SIGMA8,  color="r", ls="--", label=f"True σ₈ = {TRUE_SIGMA8}")
plt.axvline(best_s8_lpt,  color="g", ls="-.",
            label=f"Best-fit σ₈ = {best_s8_lpt:.3f}")
plt.xlabel(r"$\sigma_8$")
plt.ylabel(r"$\Delta \log \mathcal{L}$")
plt.title("σ8 recovery – LPT forward model")
plt.legend()
plt.tight_layout()
plt.show()

print(f"True  fσ₈ = {fsigma8_true:.4f}")
print(f"Fit   fσ₈ = {fs8_lpt:.4f}")

## 7. Full N-body simulation

Replacing `method='lpt'` with `method='nbody'` evolves the simulation with the full
particle-mesh N-body ODE integrator (`diffrax` backend).  This is more accurate at
the cost of longer run time.

> **Tip:** Tighten `ode_rtol` / `ode_atol` to `1e-5` for a production run.
> On GPU this runs ~10× faster than on CPU.

In [ ]:
# Build N-body likelihood with the same data and positions
lik_nbody = likelihood.VelocityFieldLikelihood(
    data_vector=DataVel,
    positions_cartesian=positions,
    mesh_shape=MESH_SHAPE,
    box_size=BOX_SIZE,
    seed=SEED,
    method="nbody",
    fixed_cosmo_params={"omega_m": TRUE_OMEGA_M},
    ode_rtol=1e-3,  # loose tolerances for the demo
    ode_atol=1e-3,
)

val_nbody_true = float(lik_nbody({"sigma8": TRUE_SIGMA8}))
print(f"N-body neg-log-lik at true σ8 = {val_nbody_true:.3f}")

# Gradient at the true cosmology
grad_nbody = float(jax.grad(lambda s8: lik_nbody({"sigma8": s8}))(TRUE_SIGMA8))
print(f"N-body gradient d(-log L)/d(σ8) at true σ8 = {grad_nbody:.4f}")

In [ ]:
fitter_nbody = SimulationFitter(
    likelihood=lik_nbody,
    initial_params={"sigma8": 0.7},
    bounds=({"sigma8": 0.2}, {"sigma8": 1.6}),
    solver="LBFGSB",
    maxiter=20,
)

best_nbody   = fitter_nbody.run()
best_s8_nbody = float(best_nbody["sigma8"])

cosmo_nbody  = generate.get_cosmology(omega_m=TRUE_OMEGA_M, sigma8=best_s8_nbody)
fs8_nbody    = float(generate.compute_fsigma8(cosmo_nbody, a=1.0))

print("\n=== Best-fit results (N-body forward model) ===")
print(f"  Best-fit σ8     = {best_s8_nbody:.4f}  (true: {TRUE_SIGMA8:.4f})")
print(f"  Best-fit f*σ8   = {fs8_nbody:.4f}  (true: {fsigma8_true:.4f})")
print(f"  Relative error = {abs(best_s8_nbody - TRUE_SIGMA8) / TRUE_SIGMA8 * 100:.1f}%")

In [ ]:
methods    = ["LPT", "N-body"]
fs8_values = [fs8_lpt, fs8_nbody]

plt.figure(figsize=(5, 4))
plt.bar(methods, fs8_values, width=0.4, color=["steelblue", "coral"])
plt.axhline(fsigma8_true, color="r", ls="--", label=f"True fσ₈ = {fsigma8_true:.4f}")
plt.ylabel(r"$f\sigma_8$")
plt.title(r"Best-fit $f\sigma_8$ by method")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  True  f*σ8 = {fsigma8_true:.4f}")
print(f"  LPT   f*σ8 = {fs8_lpt:.4f}")
print(f"  Nbody f*σ8 = {fs8_nbody:.4f}")

## 8. Summary

| Method | Best-fit σ8 | Best-fit fσ₈ | True fσ₈ |
|--------|------------|--------------|----------|
| LPT    | (see above) | (see above) | (see above) |
| N-body | (see above) | (see above) | (see above) |

*(Values are filled in after running the cells above.)*

**Tips for a science run:**
* Increase `MESH_SHAPE` to `(128, 128, 128)` or larger.
* Use `method='nbody'` with `ode_rtol=1e-5`, `ode_atol=1e-5`.
* Fix `omega_m` from CMB/BAO or jointly fit `omega_m` and `sigma8`.
* Increase `maxiter` in `SimulationFitter` until convergence.
* Use a real galaxy peculiar velocity catalogue.